### Ноутбук "EDA и подготовка данных"

#### Описание

Генерация и загрузка синтетических данных в PostgreSQL, первичная проверка целостности перед основным анализом.

##### Импорт модулей и библиотек

In [1]:
import sys
sys.path.append("../src")

from generate_data import generate_users, generate_subscription, generate_payments

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import text

from db_connection import get_engine
engine = get_engine()

##### Создание таблицы "users"

Генерация датафрейма "users_df" с 3000 пользователей, без столбца "id"

In [3]:
users_df = generate_users(3000)
display(users_df.head())
display(users_df.info())

,acquisition_channel,plan,country,signup_date
0,paid_search,free,Russia,2026-07-06
1,email,pro,Russia,2025-12-05
2,paid_search,free,Russia,2025-08-31
3,organic,basic,Russia,2026-07-09
4,paid_search,free,China,2025-07-08


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   acquisition_channel  3000 non-null   str   
 1   plan                 3000 non-null   str   
 2   country              3000 non-null   str   
 3   signup_date          3000 non-null   object
dtypes: object(1), str(3)
memory usage: 93.9+ KB


None

*Очистка таблицы "users"*

In [4]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE users RESTART IDENTITY CASCADE;"))

Генерация датафрейма "users_db" со столбцом "id"

In [5]:
users_df.to_sql("users", engine, if_exists="append", index=False)

users_db = pd.read_sql("SELECT * FROM users;", engine)
display(users_db.columns)
display(users_db.shape)

Index(['id', 'acquisition_channel', 'plan', 'country', 'signup_date'], dtype='str')

(3000, 5)

##### Создание таблицы "subscription"

Генерация датафрейма "subscriptions_df" без столбца "id"

In [6]:
subscriptions_df = generate_subscription(users_db)
display(subscriptions_df.head())
display(subscriptions_df.info())

,user_id,plan,price,start_date,end_date,status
0,2,pro,250.0,2025-12-06,2026-02-15,canceled
1,4,basic,100.0,2026-07-15,NaT,active
2,7,pro,250.0,2025-12-20,NaT,active
3,10,basic,100.0,2025-10-26,NaT,active
4,14,pro,250.0,2026-07-19,NaT,active


<class 'pandas.DataFrame'>
RangeIndex: 1336 entries, 0 to 1335
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1336 non-null   int64         
 1   plan        1336 non-null   str           
 2   price       1336 non-null   float64       
 3   start_date  1336 non-null   datetime64[us]
 4   end_date    386 non-null    datetime64[us]
 5   status      1336 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(1), str(2)
memory usage: 62.8 KB


None

Генерация датафрейма "subscriptions_db" со столбцом "id"

In [7]:
subscriptions_df.to_sql("subscriptions", engine, if_exists="append", index=False)

subscriptions_db = pd.read_sql("SELECT * FROM subscriptions;", engine)
display(subscriptions_db.columns)
display(subscriptions_db.shape)

Index(['id', 'user_id', 'plan', 'price', 'start_date', 'end_date', 'status'], dtype='str')

(1336, 7)

##### Создание таблицы "payments"

Генерация датафрейма "payments_df" без столбца "id"

In [8]:
payments_df = generate_payments(subscriptions_db)
display(payments_df.head())
display(payments_df.info())

,subscription_id,amount,payment_date,status
0,1,250.0,2025-12-06,succeeded
1,1,250.0,2026-01-05,succeeded
2,1,250.0,2026-02-04,succeeded
3,2,100.0,2026-07-15,succeeded
4,2,100.0,2026-08-14,succeeded


<class 'pandas.DataFrame'>
RangeIndex: 10746 entries, 0 to 10745
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  10746 non-null  int64         
 1   amount           10746 non-null  float64       
 2   payment_date     10746 non-null  datetime64[us]
 3   status           10746 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 335.9 KB


None

Генерация датафрейма "payments_df" со столбцом "id"

In [10]:
payments_df.to_sql("payments", engine, if_exists="append", index=False)

payments_db = pd.read_sql("SELECT * FROM payments;", engine)
display(payments_db.columns)
display(payments_db.shape)

Index(['id', 'subscription_id', 'amount', 'payment_date', 'status'], dtype='str')

(10746, 5)